<a href="https://colab.research.google.com/github/MengOonLee/LLM/blob/main/References/LangChain/LangChainAcademy/LangChain/Foundation/AdvancedAgent/ipynb/2.2_state.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%bash
pip install --no-cache-dir -qU \
    langchain langgraph langchain-core langchain-community \
    langchain-openai langchain-mcp-adapters tavily-python mcp

In [1]:
import warnings
warnings.filterwarnings("ignore")
import dotenv

_ = dotenv.load_dotenv(dotenv_path=".env", override=True)

## Write to state

In [2]:
import os
import time
from langchain import chat_models, agents, tools, messages
from langgraph import types
from langgraph.checkpoint import memory

llm = chat_models.init_chat_model(
    model="gpt-oss:120b-cloud",
    model_provider="openai",
    base_url="https://ollama.com/v1",
    api_key=os.environ["OLLAMA_API_KEY"],
    temperature=0
)

class CustomState(agents.AgentState):
    favourite_colour: str

@tools.tool
def update_favourite_colour(favourite_colour: str,
    runtime: tools.ToolRuntime) -> types.Command:
    """
    Update the favourite colour of the user in the state
    once they've revealed it.
    """
    return types.Command(update={
        "favourite_colour": favourite_colour,
        "messages": [messages.ToolMessage(
            content="Successfully updated favourite colour",
            tool_call_id=runtime.tool_call_id
        )]}
    )

agent = agents.create_agent(
    model=llm,
    tools=[update_favourite_colour],
    checkpointer=memory.InMemorySaver(),
    state_schema=CustomState
)

start_time = time.time()
response = agent.invoke(
    input={"messages": [messages.HumanMessage(content="""
        My favourite colour is green.
    """)]},
    config={"configurable": {"thread_id": "1"}}
)
end_time = time.time() - start_time
print(f"Time taken: %.2fs"%(end_time))

for m in response["messages"]:
    m.pretty_print()

Time taken: 2.19s
================================ Human Message =================================


        My favourite colour is green.
    
================================== Ai Message ==================================
Tool Calls:
  update_favourite_colour (call_kfjk49kr)
 Call ID: call_kfjk49kr
  Args:
    favourite_colour: green
================================= Tool Message =================================
Name: update_favourite_colour

Successfully updated favourite colour
================================== Ai Message ==================================

Got it! I’ve noted that your favourite colour is **green**. 🌿 Let me know if there’s anything else you’d like to share or talk about!


In [3]:
import os
import time
from langchain import chat_models, agents, tools, messages
from langgraph import types
from langgraph.checkpoint import memory

llm = chat_models.init_chat_model(
    model="gpt-oss:120b-cloud",
    model_provider="openai",
    base_url="https://ollama.com/v1",
    api_key=os.environ["OLLAMA_API_KEY"],
    temperature=0
)

class CustomState(agents.AgentState):
    favourite_colour: str

@tools.tool
def update_favourite_colour(favourite_colour: str,
    runtime: tools.ToolRuntime) -> types.Command:
    """
    Update the favourite colour of the user in the state
    once they've revealed it.
    """
    return types.Command(update={
        "favourite_colour": favourite_colour,
        "messages": [messages.ToolMessage(
            content="Successfully updated favourite colour",
            tool_call_id=runtime.tool_call_id
        )]}
    )

agent = agents.create_agent(
    model=llm,
    tools=[update_favourite_colour],
    checkpointer=memory.InMemorySaver(),
    state_schema=CustomState
)

start_time = time.time()
response = agent.invoke(
    input={
        "messages": [messages.HumanMessage(content="""
            Hello, how are you?
        """)],
        "favourite_colour": "green"
        },
    config={"configurable": {"thread_id": "10"}}
)
end_time = time.time() - start_time
print(f"Time taken: %.2fs"%(end_time))

for m in response["messages"]:
    m.pretty_print()

Time taken: 1.20s
================================ Human Message =================================


            Hello, how are you?
        
================================== Ai Message ==================================

Hello! I’m doing great, thank you for asking. How can I help you today?


## Read state

In [4]:
import os
import time
from langchain import chat_models, agents, tools, messages
from langgraph import types
from langgraph.checkpoint import memory

llm = chat_models.init_chat_model(
    model="gpt-oss:120b-cloud",
    model_provider="openai",
    base_url="https://ollama.com/v1",
    api_key=os.environ["OLLAMA_API_KEY"],
    temperature=0
)

class CustomState(agents.AgentState):
    favourite_colour: str

@tools.tool
def update_favourite_colour(favourite_colour: str,
    runtime: tools.ToolRuntime) -> types.Command:
    """
    Update the favourite colour of the user in the state
    once they've revealed it.
    """
    return types.Command(update={
        "favourite_colour": favourite_colour,
        "messages": [messages.ToolMessage(
            content="Successfully updated favourite colour",
            tool_call_id=runtime.tool_call_id
        )]}
    )

@tools.tool
def read_favourite_colour(runtime: tools.ToolRuntime) -> str:
    """
    Read the favourite colour of the user from the state.
    """
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

agent = agents.create_agent(
    model=llm,
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=memory.InMemorySaver(),
    state_schema=CustomState
)

start_time = time.time()
response = agent.invoke(
    input={"messages": [messages.HumanMessage(content="""
        My favourite colour is green.
    """)]},
    config={"configurable": {"thread_id": "1"}}
)
end_time = time.time() - start_time
print(f"Time taken: %.2fs"%(end_time))

for m in response["messages"]:
    m.pretty_print()

Time taken: 3.16s
================================ Human Message =================================


        My favourite colour is green.
    
================================== Ai Message ==================================
Tool Calls:
  update_favourite_colour (call_j30fx5iz)
 Call ID: call_j30fx5iz
  Args:
    favourite_colour: green
================================= Tool Message =================================
Name: update_favourite_colour

Successfully updated favourite colour
================================== Ai Message ==================================

Got it—your favourite colour is now set to **green**. Let me know if you’d like to change it or need anything else!


In [5]:
import time
from langchain import messages

start_time = time.time()
response = agent.invoke(
    input={"messages": [messages.HumanMessage(content="""
        What's my favourite colour?
    """)]},
    config={"configurable": {"thread_id": "1"}}
)
end_time = time.time() - start_time
print(f"Time taken: %.2fs"%(end_time))

for m in response["messages"]:
    m.pretty_print()

Time taken: 1.87s
================================ Human Message =================================


        My favourite colour is green.
    
================================== Ai Message ==================================
Tool Calls:
  update_favourite_colour (call_j30fx5iz)
 Call ID: call_j30fx5iz
  Args:
    favourite_colour: green
================================= Tool Message =================================
Name: update_favourite_colour

Successfully updated favourite colour
================================== Ai Message ==================================

Got it—your favourite colour is now set to **green**. Let me know if you’d like to change it or need anything else!
================================ Human Message =================================


        What's my favourite colour?
    
================================== Ai Message ==================================
Tool Calls:
  read_favourite_colour (call_z12byrbl)
 Call ID: call_z12byrbl
  Args:
========================